In [63]:
from enum import Enum
from collections import deque
from dataclasses import dataclass, field, InitVar
from typing import Deque, List, Set, Dict, Tuple


class PacketType(Enum):    
    EXCESS = 0
    DEFICIT = 1
    BALANCED = 2
    UNDEFINED = 3


@dataclass
class EnergyPacket:
    capacity: float
    energy: float
        
    @property
    def capacity_max(self) -> float:
        return self.capacity + self.energy



@dataclass
class PhasePair:
    """ A phase pair consists of excess energy packets and deficit energy packets belonging to exaxtly one excess phase and one deficit phase.
    """
    energy_packets: Dict[PacketType, Deque[EnergyPacket]] = field(default_factory=lambda: {PacketType.EXCESS: deque(), PacketType.DEFICIT: deque()})
    n_unbalanced: Dict[PacketType, int] = field(default_factory = lambda: {PacketType.EXCESS: 0, PacketType.DEFICIT: 0})
    N_unbalanced_total: int = 0

    energy_excess_initial: InitVar[float | None] = None
    energy_deficit_initial: InitVar[float | None] = None
    
    def __post_init__(self, energy_excess_initial: float, energy_deficit_initial: float):
        self.insert_packet(
            index_packet=0,
            packet_type=PacketType.EXCESS,
            energy_packet=EnergyPacket(capacity=0, energy=energy_excess_initial)
        )
        self.insert_packet(
            index_packet=0,
            packet_type=PacketType.DEFICIT,
            energy_packet=EnergyPacket(capacity=0, energy=energy_deficit_initial)
        )
        
    
    @property
    def phase_type(self):
        if self.n_unbalanced[PacketType.EXCESS] == 0 and self.n_unbalanced[PacketType.DEFICIT] == 0:
            return PacketType.BALANCED
            
        if self.n_unbalanced[PacketType.EXCESS] >= 1 and self.n_unbalanced[PacketType.DEFICIT] >= 1:
            return PacketType.UNDEFINED

        if self.n_unbalanced[PacketType.EXCESS] >= 1:
            return PacketType.EXCESS

        if self.n_unbalanced[PacketType.DEFICIT] >= 1:
            return PacketType.DEFICIT

        raise


    def remove_packet(self, index_packet: int, packet_type: PacketType ):
        """Removes an energy packet of packet_type at index_packet from the phase pair.
        Rotate left by index_packet is O(index_packet), pop left O(1) and rotate right by index_packet O(index_packet)
        """
        self.energy_packets[packet_type].rotate(-index_packet)
        self.energy_packets[packet_type].popleft()
        self.energy_packets[packet_type].rotate( index_packet)

        self.n_unbalanced[packet_type] -= 1  # reduce the number of unbalanced packets
        self.N_unbalanced_total -= 1

    
    def insert_packet(self, index_packet: int, packet_type: PacketType, energy_packet: EnergyPacket ):
        """Inserts an energy packet of packet_type at index_packet to the phase pair.
        Rotate left by index_packet is O(index_packet), append left O(1) and rotate right by index_packet O(index_packet)
        """
        self.energy_packets[packet_type].rotate(-index_packet)
        self.energy_packets[packet_type].appendleft(energy_packet)
        self.energy_packets[packet_type].rotate( index_packet)
    
        self.n_unbalanced[packet_type] += 1  # increase the number of unbalanced packets
        self.N_unbalanced_total += 1

    
    def balance_packet(self):
        self.n_unbalanced[PacketType.EXCESS] -= 1
        self.n_unbalanced[PacketType.DEFICIT] -= 1
        self.N_unbalanced_total -= 2

        # rotate the packet-tuples left to move the now balanced packet to the end of the deque            
        self.energy_packets[PacketType.EXCESS].rotate(-1)
        self.energy_packets[PacketType.DEFICIT].rotate(-1)


    def merge_packets(self, packet_type: PacketType):
        while self.n_unbalanced[packet_type] > 1 and self.energy_packets[packet_type][0].capacity_max >= self.energy_packets[packet_type][1].capacity:
            print(f'Merging packets of type {packet_type}')
            print(f'Before: {self.energy_packets[packet_type]}')
            self.energy_packets[packet_type][0].energy += self.energy_packets[packet_type][1].energy  # combine the energy content
    
            # remove the merged packet
            self.remove_packet(
                index_packet=1,
                packet_type=packet_type
            )
            
            print(f'Now: {self.energy_packets[packet_type]}')

    
    def split_packet(self, index_packet: int, packet_type: PacketType, capacity_to_split: float):
        if capacity_to_split <= self.energy_packets[packet_type][index_packet].capacity or self.energy_packets[packet_type][index_packet].capacity_max <= capacity_to_split:
            return  # we can only split a packet, when the capacity value lies within the packet
    
        print(f'Splitting packet of type {packet_type} at {capacity_to_split}')
        print(f'Before: {self.energy_packets[packet_type]}')
        
        energy_new_packet = self.energy_packets[packet_type][index_packet].capacity_max - capacity_to_split  # calculate the remaining energy content
        self.energy_packets[packet_type][index_packet].energy = capacity_to_split - self.energy_packets[packet_type][index_packet].capacity  # calculate the lower packets energy content.
        
        # insert the new packet at index 1
        self.insert_packet(
            index_packet=index_packet+1,
            packet_type=packet_type,
            energy_packet=EnergyPacket(
                capacity=capacity_to_split, 
                energy=energy_new_packet
            )
        )
        
        print(f'Now: {self.energy_packets[packet_type]}')
    
    

@dataclass
class PhaseGroup:
    """
    A phase-group is collection of phase-pairs that can be compressed, balanced, and combined with other phase groups.
    A phase-group of type UNDEFINED will need to be balanced first and will then be either EXCESS, DEFICIT, or BALANCED.
    Phase-groups of type EXCESS or DEFICIT can be compressed by shifting the energy packets of the grouped phase pairs.
    """
    phase_pairs: Deque[PhasePair] = field(repr=False)
    
    group_type: PacketType
    index_start: int
    index_end: int = None
    
    index_target: int = None
    indices_to_shift: List[int] = field(default_factory=list)
    capacities_for_shift: List[int] = field(default_factory=list)


    def balance_group(self):
        """
        Balancing a group is only required for group_type==UNDEFINED and will only need to check the very first phase pair at index_start.
        """
        if self.group_type != PacketType.UNDEFINED:
            print(f'A group of type {self.group_type} cannot be balanced!')
            raise

        while self.phase_pairs[self.index_start].n_unbalanced[PacketType.EXCESS] > 0 and self.phase_pairs[self.index_start].n_unbalanced[PacketType.DEFICIT] > 0:            
            # The earliest available capacity is the maximum of both packets. Raise packets to the earliest available capacity which might touch the next energy packets of the same kind.
            capacity_earliest_available = max( self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity, self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity )

            if self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity < capacity_earliest_available:
                self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity = capacity_earliest_available
                self.phase_pairs[self.index_start].merge_packets(PacketType.EXCESS)  # Merge and potentially remove packets.

            if self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity < capacity_earliest_available:
                self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity = capacity_earliest_available
                self.phase_pairs[self.index_start].merge_packets(PacketType.DEFICIT)  # Merge and potentially remove packets.

            # Balance excess and deficit by computing the minimum of both energy contents and create a new packet with the remaining content.
            capacity_max_excess = self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity_max
            capacity_max_deficit = self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity_max
            capacity_to_split = min( capacity_max_excess, capacity_max_deficit )

            if capacity_max_excess > capacity_to_split:  # excess remaining
                # Split will create a new unbalanced energy packet
                self.phase_pairs[self.index_start].split_packet(index_packet=0, packet_type=PacketType.EXCESS, capacity_to_split=capacity_to_split)  
            elif capacity_max_deficit > capacity_to_split:  # deficit remaining
                # Split will create a new unbalanced energy packet
                self.phase_pairs[self.index_start].split_packet(index_packet=0, packet_type=PacketType.DEFICIT, capacity_to_split=capacity_to_split)  

            self.phase_pairs[self.index_start].balance_packet()

        
        self.group_type = self.phase_pairs[self.index_start].phase_type
        
        
        match self.group_type:
            case PacketType.BALANCED:
                self.indices_to_shift = [None]
                self.capacities_for_shift = [self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][-1].capacity_max]
            case PacketType.EXCESS:
                self.indices_to_shift = [self.index_start]
                self.capacities_for_shift = [self.phase_pairs[self.index_start].energy_packets[PacketType.EXCESS][0].capacity]
            case PacketType.DEFICIT:
                self.indices_to_shift = [self.index_start]
                self.capacities_for_shift = [self.phase_pairs[self.index_start].energy_packets[PacketType.DEFICIT][0].capacity]

        print('----')
        print(self)
        
    _merge_rules = {
        (PacketType.UNDEFINED, PacketType.UNDEFINED): (None, "This needs to undergo balance first."),
        (PacketType.UNDEFINED, PacketType.EXCESS)   : (None, "This needs to undergo balance first."),
        (PacketType.UNDEFINED, PacketType.DEFICIT)  : (None, "This needs to undergo balance first."),
        (PacketType.EXCESS,    PacketType.UNDEFINED): (None, "This needs to undergo balance first."),
        (PacketType.DEFICIT,   PacketType.UNDEFINED): (None, "This needs to undergo balance first."),
        
        (PacketType.BALANCED,  PacketType.BALANCED) : (PacketType.BALANCED, "Same type"),
        (PacketType.EXCESS,    PacketType.EXCESS)   : (PacketType.EXCESS, "Same type"),
        (PacketType.DEFICIT,   PacketType.DEFICIT)  : (PacketType.DEFICIT, "Same type"),
        
        (PacketType.BALANCED,  PacketType.DEFICIT)  : (PacketType.DEFICIT, "DEFICIT will be shifted left over BALANCE."),
        (PacketType.EXCESS,    PacketType.BALANCED) : (PacketType.EXCESS,"EXCESS will be shifted right over BALANCE."),     
        
        (PacketType.BALANCED,  PacketType.EXCESS)   : (None, "We would loose information for potential later shift operations."),
        (PacketType.DEFICIT,   PacketType.BALANCED) : (None, "We would loose information for potential later shift operations."),
        
        (PacketType.EXCESS,    PacketType.DEFICIT)  : (None, "This needs to undergo shifting first."),
        (PacketType.DEFICIT,   PacketType.EXCESS)   : (None, "This needs to undergo shifting first."),
    }

    
    def can_merge(self, other: 'PhaseGroup') -> bool:
        return PhaseGroup._merge_rules[(self.group_type, other.group_type)][0] is not None

    
    def merge_with(self, other: 'PhaseGroup'):
        print(f'Merging:\n  - {self} and\n  - {other}')
        
        new_type, reason = PhaseGroup._merge_rules[(self.group_type, other.group_type)]

        if new_type is None:
            return False, reason

        self.group_type = new_type
        
        """Merging two groups will allways set the end index of the first one to the end index of the second one."""
        self.index_end = other.index_end        
        self.indices_to_shift.extend(other.indices_to_shift)
        self.capacities_for_shift.extend(other.capacities_for_shift)
        
        print(f'Merge result:\n  - {self}')
        return True, reason


    def shift_excess(self, other: 'PhaseGroup'):
        """
        For EXCESS groups we iterate over the indices and capacities in reverse direction and shift the to start of the next group.
        """
        if self.group_type != PacketType.EXCESS or other.group_type != PacketType.DEFICIT:
            return False
        return True
        
        
    
    def shift_deficit(self):
        """
        For DEFICIT groups we iterate over the indices and capacities in forward direction and shift the to start of the group.
        """
        if self.group_type != PacketType.DEFICIT:
            return False

        print(f'Shift for {self}')
        
        index_target = self.index_start
        phase_pair_target =  self.phase_pairs[index_target]
        capacity_hurdle = 0
        for index, capacity in zip(self.indices_to_shift, self.capacities_for_shift):            
            if index == self.index_start:
                """Same index"""
                continue
            if index is None:
                """Nothing to shift from a BALANCED index"""
                continue

            capacity_hurdle = max(capacity_hurdle, capacity)
            print(f'Shift from {index} to {index_target}')
            
            while self.phase_pairs[index].n_unbalanced[PacketType.DEFICIT]:
                capacity_max_target = phase_pair_target.energy_packets[PacketType.DEFICIT][phase_pair_target.n_unbalanced[PacketType.DEFICIT] - 1].capacity_max     
                print('--------------')
                print(f'{capacity_max_target = }')
                print(f'{self.phase_pairs[index] = }')
                pkt = self.phase_pairs[index].energy_packets[PacketType.DEFICIT][0]
                print(f'{pkt = }')
                pkt.capacity = max(pkt.capacity, capacity_max_target, capacity_hurdle)
                phase_pair_target.insert_packet(
                    index_packet=phase_pair_target.n_unbalanced[PacketType.DEFICIT],
                    packet_type=PacketType.DEFICIT,
                    energy_packet=pkt
                )
                self.phase_pairs[index].remove_packet(
                    index_packet=0,
                    packet_type=PacketType.DEFICIT
                )
                print(f'{self.phase_pairs[index] = }')
                print(f'{phase_pair_target = }')
            
        return True
        

@dataclass
class Context:
    energy_excess_per_phase_initial: List[float]
    energy_deficit_per_phase_initial: List[float]
    N_phases: int = 0
    
    phase_pairs: Deque[PhasePair] = None  # The algorithm will store results in this one
    
    phase_groups: Deque[PhaseGroup] = None  # The algorithm will work on this one

    indices_to_balance: Deque[int] = None
    indices_first_and_last: Dict[PacketType, List[int]] = field(default_factory=lambda: {PacketType.EXCESS: [None, None], PacketType.DEFICIT: [None, None]})
    
    @property
    def N_unbalanced_total(self):
        return sum([phase_pair.N_unbalanced_total for phase_pair in self.phase_pairs])

    @property
    def n_unbalanced_excess(self):
        return sum([phase_pair.n_unbalanced[PacketType.EXCESS] for phase_pair in self.phase_pairs])

    @property
    def n_unbalanced_deficit(self):
        return sum([phase_pair.n_unbalanced[PacketType.DEFICIT] for phase_pair in self.phase_pairs])

    
    def __post_init__(self):
        assert len(self.energy_excess_per_phase_initial) == len(self.energy_deficit_per_phase_initial)
        
        self.N_phases = len(self.energy_deficit_per_phase_initial)

        self.indices_to_balance = deque(range(self.N_phases))        

        self.phase_pairs = deque([PhasePair(
            energy_excess_initial=energy_excess_initial,
            energy_deficit_initial=energy_deficit_initial,
        ) for (energy_excess_initial, energy_deficit_initial) in zip(self.energy_excess_per_phase_initial, self.energy_deficit_per_phase_initial)])
        
        self.phase_groups = deque([PhaseGroup(
            group_type=PacketType.UNDEFINED,  # A phase-group of type UNDEFINED will need to be balanced first and will then be either EXCESS, DEFICIT, or BALANCED
            index_start=index_phase,
            index_end=index_phase,
            phase_pairs=self.phase_pairs
        ) for index_phase in range(self.N_phases)])
                

    def print_phase_groups(self) -> str:
        short_dict = {}
        for pg in self.phase_groups:
            _ = {(pg.index_start, pg.index_end): pg.group_type.name}    
            short_dict.update(_)
        return str(short_dict)
        return str({key: short_dict[key] for key in sorted(short_dict.keys())})

    def short_phases(self) -> str:
        s = ''
        for pg in self.phase_groups:
            if pg.index_start == 0 or pg.index_start > pg.index_end:
                s += '|'
            s += pg.group_type.name[0]  
        return s
    
energy_excess_per_phase_initial = [3,2,2,3,3,2,6,2,4,5,7,2,2,2]
energy_deficit_per_phase_initial= [3,2,3,3,4,2,6,2,3,8,3,3,1,1]

all_possibe_combinations = 'BBBDBEDBDDDEEBEDEE'

energy_excess_per_phase_initial = [3,2,4,3,6,3,1,2,2,5,2,7,2,1,2,1,4,8]
energy_deficit_per_phase_initial= [3,2,4,5,6,2,2,2,3,8,3,5,1,1,1,2,1,5]

#energy_per_phase_initial: List[Dict[PacketType, float]] = [{
#    PacketType.EXCESS: energy_excess_initial,
#    PacketType.DEFICIT: energy_deficit_initial
#} for (energy_excess_initial, energy_deficit_initial) in zip(energy_excess_per_phase_initial, energy_deficit_per_phase_initial)]


In [2]:
"""
Packets in each phase are stored in a deque.
Whenever a packet is balanced we rotate the deque to the left by 1.
This keeps packets in order and unbalanced packets are always at the start of the deque.
"""
    
def balance(ctx):
    print(f'Balancing: {ctx}')
    it = 0 # will point to the first EXCESS phase
    """lock_i = False    
    lock_j = False # will point to the last EXCESS phase
    lock_k = False # will point to the first DEFICIT phase
    lock_o = False # will point to the last DEFICIT phase
    """
    for phase_group in ctx.phase_groups:
        phase_group.balance_group()

        """ 
        has_excess = phase_group.group_type == PacketType.EXCESS
        has_deficit = phase_group.group_type == PacketType.DEFICIT
        if has_excess:
            lock_o = False
            
        if not lock_i and has_excess:
            #i = index_phase
            ctx.indices_first_and_last[PacketType.EXCESS][0] = it
            lock_i = True

        if not lock_j and has_excess:
            #j = index_phase
            ctx.indices_first_and_last[PacketType.EXCESS][1] = it
            lock_j = True

        if has_deficit:
            lock_j = False
            
        if not lock_k and has_deficit:
            #k = index_phase
            ctx.indices_first_and_last[PacketType.DEFICIT][0] = it
            lock_k = True

        if not lock_o and has_deficit:
            #o = index_phase
            ctx.indices_first_and_last[PacketType.DEFICIT][1] = it
            lock_o = True
        
        it += 1
        
        
    print(f'{ctx.indices_first_and_last = }')
    """
    return ctx.n_unbalanced_excess == 0 or ctx.n_unbalanced_deficit == 0

    

In [47]:
def merge_groups(ctx: Context, rotate_by=None):
    if rotate_by is None:
        #if ctx.indices_first_and_last[PacketType.EXCESS][0] < ctx.indices_first_and_last[PacketType.DEFICIT][0] < ctx.indices_first_and_last[PacketType.EXCESS][1]:
        #    # rotate right to the last excess            
        #    rotate_by = ctx.N_phases - ctx.indices_first_and_last[PacketType.EXCESS][1]
        #else:
        #    # rotate left to the first excess            
        #    rotate_by = -ctx.indices_first_and_last[PacketType.EXCESS][0]
        rotate_by = 0
        
    print(f'=============== ROTATION BY {rotate_by} ===============')
    ctx.phase_groups.rotate(rotate_by)
    print(ctx.print_phase_groups())
    
    print('')

    n_rotations_total = len(ctx.phase_groups)
    n_rotated = 0
    while n_rotated < n_rotations_total:
        
        print('\n----')
        str_original_0 = ctx.short_phases()

        ctx.phase_groups[-1].can_merge(ctx.phase_groups[0])
        
        while ctx.phase_groups[-1].can_merge(ctx.phase_groups[0]):
            ctx.phase_groups.rotate(1)
            n_rotations_total += 1
            print('rotated back by 1')
        
        str_original = ctx.short_phases()
        merged, reason = ctx.phase_groups[0].merge_with(ctx.phase_groups[1])
        
        if merged:
            ctx.phase_groups.rotate(-1)
            ctx.phase_groups.popleft()
            ctx.phase_groups.rotate(1)            
            n_rotations_total -= 1
        else:            
            print(f"Can't merge: {reason}")            
            ctx.phase_groups.rotate(-1)
            n_rotated += 1

        str_after = ctx.short_phases()

        print(f'{str_original_0} -> {str_original} -> {str_after}  ({"same length" if len(str_original) == len(str_after) else "1 merged"})')
        
    print(ctx.print_phase_groups())
    print('')
    

In [64]:
def shift_groups(ctx: Context):
    """Iterate over all phase_groups and shift EXCESS groups to the next DEFICIT group"""
    #ctx.phase_groups[0].shift_excess(ctx.phase_groups[1])
    for grp in ctx.phase_groups:
        grp.shift_deficit()

ctx = Context(energy_excess_per_phase_initial=energy_excess_per_phase_initial, energy_deficit_per_phase_initial=energy_deficit_per_phase_initial)

done = balance(ctx)
if done:
    print('We are done :-)')
merge_groups(ctx)
shift_groups(ctx)


Balancing: Context(energy_excess_per_phase_initial=[3, 2, 4, 3, 6, 3, 1, 2, 2, 5, 2, 7, 2, 1, 2, 1, 4, 8], energy_deficit_per_phase_initial=[3, 2, 4, 5, 6, 2, 2, 2, 3, 8, 3, 5, 1, 1, 1, 2, 1, 5], N_phases=18, phase_pairs=deque([PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=3)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=3)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=2)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=2)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=4)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=4)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total

In [50]:
rotate_by = None
ctx = Context(energy_excess_per_phase_initial=energy_excess_per_phase_initial, energy_deficit_per_phase_initial=energy_deficit_per_phase_initial)

done = balance(ctx)
if done:
    print('We are done :-)')

#ctx.indices_first_and_last[PacketType.EXCESS][0] = 0
print('')
ref_orders      = [
    {(4, 4): 'BALANCED', (5, 5): 'EXCESS', (6, 10): 'DEFICIT', (11, 14): 'EXCESS', (15, 15): 'DEFICIT'},
    {(3, 3): 'DEFICIT', (16, 2): 'EXCESS'},
    {(2, 3): 'DEFICIT', (16, 1): 'EXCESS' },
    {(1, 3): 'DEFICIT', (16, 0): 'EXCESS' },
    {(0, 3): 'DEFICIT', (16, 17): 'EXCESS'},
]

merge_groups(ctx, rotate_by=rotate_by)
print('')
current_order = {}
for pg in ctx.phase_groups:
    _ = {(pg.index_start, pg.index_end): pg.group_type.name}    
    current_order.update(_)

print('\n==== Current and ref ====')
print(ctx.print_phase_groups())
# check for equivalence

for key, value in current_order.items():
    match = False
    for ref_order in ref_orders:            
        if key in ref_order:
            assert value == ref_order[key]
            match = True
            continue
    assert match

Balancing: Context(energy_excess_per_phase_initial=[3, 2, 4, 3, 6, 3, 1, 2, 2, 5, 2, 7, 2, 1, 2, 1, 4, 8], energy_deficit_per_phase_initial=[3, 2, 4, 5, 6, 2, 2, 2, 3, 8, 3, 5, 1, 1, 1, 2, 1, 5], N_phases=18, phase_pairs=deque([PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=3)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=3)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=2)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=2)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total=2), PhasePair(energy_packets={<PacketType.EXCESS: 0>: deque([EnergyPacket(capacity=0, energy=4)]), <PacketType.DEFICIT: 1>: deque([EnergyPacket(capacity=0, energy=4)])}, n_unbalanced={<PacketType.EXCESS: 0>: 1, <PacketType.DEFICIT: 1>: 1}, N_unbalanced_total